# Course 19 — Protocol Composition Security

Build and attack a trusted A2A → application adapter → MCP support workflow. The agent runtime proposes; application code authenticates, authorizes, exchanges credentials, verifies contracts, executes, reconciles, and records evidence.

## Objectives and safety

We will preserve subject, actor, tenant, audience, action, resource, purpose, approval, lifecycle, operation, error, cancellation, and trace semantics. This lab is offline and uses training keys plus process-local stores. It does **not** prove TLS, process isolation, production OAuth, durable storage, or deployment security.

In [ ]:
from dataclasses import asdict, replace
from pathlib import Path
import runpy
import jwt

root = Path.cwd().resolve()
while root != root.parent and not (root / 'pyproject.toml').exists():
    root = root.parent
assert (root / 'pyproject.toml').exists(), 'Run from inside the repository'
course = root / 'curriculum/advanced/19-protocol-composition-security'
lab = runpy.run_path(course / 'lab.py')
print('Loaded', course.relative_to(root))

In [ ]:
{name: lab[name] for name in ('A2A_PROTOCOL_VERSION', 'MCP_PROTOCOL_VERSION', 'SOURCE_AUDIENCE', 'TARGET_AUDIENCE', 'POLICY_VERSION')}

## 1. Name every boundary

The runtime emits untrusted intent. The A2A transport authenticates a sender workload. The adapter is a policy enforcement point and token-exchange client. The MCP server owns the final resource effect and authorizes again. A success response is not the effect oracle; the target ledger/datastore is.

In [ ]:
read_env = lab['reviewed_environment'](action='read', operation_id='nb-read')
asdict(read_env.peer), asdict(read_env.parent), asdict(read_env.proposal)

In [ ]:
message = read_env.message
{
    'protobuf_type': message.DESCRIPTOR.full_name,
    'message_id': message.message_id,
    'payload': lab['message_payload'](message),
    'metadata_is_untrusted': dict(message.metadata),
}

### Identity is not message content

The tenant hint and authorization string above are deliberately untrusted. Trusted tenant and sender facts come from the authenticated peer plus parent authority. The Agent Card and message describe; they do not authorize.

In [ ]:
context = read_env.adapter.normalize(
    read_env.peer, read_env.parent, read_env.proposal, read_env.message, now=read_env.now
)
asdict(context)

## 2. Exchange instead of forwarding

The source grant is accepted only by the adapter audience. The adapter verifies exact message/context bindings and issues a shorter-lived token for the MCP target. We decode the training tokens for inspection; never log real bearer tokens.

In [ ]:
source_claims = lab['decode_grant'](read_env.source_token, lab['SOURCE_AUDIENCE'], now=read_env.now)
target_token = read_env.adapter.exchange.exchange(
    read_env.source_token, context, read_env.message, now=read_env.now
)
target_claims = lab['decode_grant'](target_token, lab['TARGET_AUDIENCE'], now=read_env.now)
{
    'source': {k: source_claims[k] for k in ('aud', 'sub', 'actor', 'tenant', 'action', 'resource', 'jti')},
    'target': {k: target_claims[k] for k in ('aud', 'sub', 'actor', 'act', 'tenant', 'action', 'resource', 'parent_jti')},
    'tokens_are_distinct': read_env.source_token != target_token,
}

### Check the subset relation

The target keeps the human subject, changes the current actor to the adapter, retains the prior actor in history, names `mcp://support`, and carries only the exact ticket action/resource/purpose. RFC 8693 defines exchange mechanics; authorization-server policy must enforce this narrowing.

In [ ]:
read_outcome = await read_env.adapter.compose(
    peer=read_env.peer, parent=read_env.parent, proposal=read_env.proposal,
    message=read_env.message, source_token=read_env.source_token, now=read_env.now
)
asdict(read_outcome)

## 3. Verify the MCP contract before invocation

The previous cell used a real official MCP in-memory client: negotiated protocol metadata, `tools/list`, semantic contract comparison, and `tools/call`. In-memory transport does not prove network or process controls.

In [ ]:
lab['REVIEWED_TOOL_VIEWS']

In [ ]:
drift_env = lab['reviewed_environment'](operation_id='nb-drift')
drift_env.adapter.reviewed_tools = {**lab['REVIEWED_TOOL_VIEWS'], 'ticket.admin': {}}
try:
    await drift_env.adapter.compose(
        peer=drift_env.peer, parent=drift_env.parent, proposal=drift_env.proposal,
        message=drift_env.message, source_token=drift_env.source_token, now=drift_env.now
    )
except lab['CompositionDenied'] as error:
    print(error.code, 'effects:', drift_env.target.effect_ledger)

## 4. Bind approval to one exact effect

A write receipt binds subject, tenant, action, resource, operation, canonical argument digest, policy version, approver, and lifetime. The adapter prechecks it; the target consumes it immediately before the effect. An approval ID in a tool argument would not be evidence.

In [ ]:
comment_env = lab['reviewed_environment'](action='comment', operation_id='nb-comment')
receipt = comment_env.approvals._receipts[comment_env.proposal.approval_id]
{k: v for k, v in asdict(receipt).items() if k != 'signature'}

In [ ]:
comment_outcome = await comment_env.adapter.compose(
    peer=comment_env.peer, parent=comment_env.parent, proposal=comment_env.proposal,
    message=comment_env.message, source_token=comment_env.source_token, now=comment_env.now
)
{
    'outcome': asdict(comment_outcome),
    'effect_count': len(comment_env.target.comments),
    'approval_consumed': comment_env.approvals._receipts[comment_env.proposal.approval_id].consumed,
}

## 5. Preserve failure and lifecycle semantics

A denial is permanent, a known pre-effect transient can be retried within budget, and a lost response after possible commit is unknown—not failed and not successful. Reconcile unknown outcomes before retrying. Cancellation is cooperative and cannot erase a committed effect.

In [ ]:
for code, effect_possible in [
    ('TARGET_TENANT_DENIED', False), ('TRACE_CONTEXT_INVALID', False),
    ('UPSTREAM_UNAVAILABLE', False), ('CONNECTION_LOST', True), ('NEW_CODE', False),
]:
    outcome = lab['classify_failure'](code, effect_possible=effect_possible)
    print(code, '→', outcome.state, 'retryable=', outcome.retryable, 'mapped=', outcome.code)

In [ ]:
unknown_env = lab['reviewed_environment'](action='comment', operation_id='nb-unknown')
unknown = await unknown_env.adapter.compose(
    peer=unknown_env.peer, parent=unknown_env.parent, proposal=unknown_env.proposal,
    message=unknown_env.message, source_token=unknown_env.source_token, now=unknown_env.now,
    simulate='response-loss-after-effect'
)
unknown.state, unknown.retryable, len(unknown_env.target.comments)

In [ ]:
reconciled = await unknown_env.adapter.compose(
    peer=unknown_env.peer, parent=unknown_env.parent, proposal=unknown_env.proposal,
    message=unknown_env.message, source_token=unknown_env.source_token, now=unknown_env.now
)
reconciled.state, reconciled.code, reconciled.reconciled, len(unknown_env.target.comments)

The write committed before the simulated response loss. Reconciliation returned the recorded result, and the target still has exactly one comment. A new attempt ID would be useful for diagnostics, but the logical operation ID and request digest must remain stable.

In [ ]:
cancel_env = lab['reviewed_environment'](action='comment', operation_id='nb-cancel')
cancel_env.adapter.cancel(cancel_env.proposal.operation_id)
cancelled = await cancel_env.adapter.compose(
    peer=cancel_env.peer, parent=cancel_env.parent, proposal=cancel_env.proposal,
    message=cancel_env.message, source_token=cancel_env.source_token, now=cancel_env.now
)
cancelled.state, cancelled.retryable, cancel_env.target.comments

## 6. Correlate without authorizing

W3C trace context joins evidence. It does not carry trusted user, tenant, scope, or approval. Preserve the trace ID, create a new span per hop, strip/limit baggage, and keep security-critical events on a durable unsampled path.

In [ ]:
[(item['hop'], item['traceparent'].split('-')[1], item['traceparent'].split('-')[2]) for item in read_outcome.trace]

## 7. Run the labelled campaign

Positive controls expose a system that blocks everything. Negative and boundary cases expose unsafe allows. Failure injection exposes retry and outcome bugs. Independent effect counts expose response-only oracles.

In [ ]:
demo = await lab['run_demo_async']()
demo['campaign']

## Production delta

Replace training JWTs with an organizational authorization server and resource indicators; in-memory identity with mTLS/SPIFFE plus reviewed peer records; process-local approval/idempotency state with durable atomic stores; in-memory MCP with authenticated network transport and isolation; hard-coded policy with governed OPA/Cedar/application policy; and direct dictionaries with OpenTelemetry plus independent effect evidence. Add bounded retries, circuit breaking, revocation propagation, key rotation, privacy controls, and incident runbooks.

## Exercises

1. Add an `input_required` state that only trusted user input can resume.
2. Add a second tenant and prove non-enumerating isolation at lookup, cache, and reconciliation.
3. Add a deliberate tool schema change and require explicit contract review.
4. Model cancellation after commit and a separately approved compensation.
5. Compare the composition with a direct ticket API and quantify coordination tax.

## Reflection

- Which fact in this workflow is easiest to copy across protocols but hardest to prove?
- Where does your production system distinguish a denial, a domain error, and an unknown outcome?
- What independent evidence proves that a consequential effect did—or did not—occur?
- Which component owns reconciliation after a crash, and how long are its idempotency records retained?
- Could removing one protocol or intermediary reduce more risk than adding another control?